# Day 2 · 01 · Power BI Semantic Model on Databricks

![Power BI report mockup](../../assets/images/powerbi_report_mockup.png)

This is the official Power BI semantic-model module. Day 1 built the Gold model; this notebook decides how Power BI should consume it.

## Program Map for Power BI Developers

Session 2 starts where Power BI developers usually feel the business impact: model shape, refresh behavior, query cost and ownership.

| Module | Status | Goal for Power BI developers | End artifact |
|---|---|---|---|
| Power BI semantic model | Required | choose source tables, relationships, measures and refresh mode | Power BI handoff packet |
| Workshop 2 dataset performance | Required | build serving objects that reduce report cost and support incremental refresh | monthly aggregate, incremental view, KPI table |
| Performance and automation orientation | Required | troubleshoot slow Databricks-backed reports and understand production refresh | operations and certification checklist |
| AI/BI Dashboards and Genie | Optional | compare Databricks-native BI with Power BI for governed exploration | AI/BI checklist |

Expected observation: Day 2 is about consumption design. The same Gold model can produce a fast, trusted semantic model or a slow, expensive one depending on these decisions.

## Business Scenario

RetailHub wants to publish an executive Sales dashboard and a drill-through detail page. The Gold model is ready, but the team must still decide source tables, relationships, measures, refresh mode and handoff responsibilities.

## Power BI Developer Lens

The central question is not “can Power BI connect to Databricks?” It can. The professional question is: **which Databricks object should Power BI consume, under which refresh mode, with which KPI ownership and cost guardrails?**

Keep these decisions explicit:

- model shape: star schema, wide table or serving aggregate,
- measure ownership: Databricks Gold field, Databricks KPI table or DAX measure,
- refresh mode: Import, DirectQuery or Composite,
- troubleshooting path: Performance Analyzer -> SQL Warehouse Query History -> Query Profile,
- ownership: who changes the source object, KPI definition and refresh schedule.

## Learning Objectives

By the end of this notebook you can:

- define a Power BI semantic model and its Databricks dependency,
- choose between star schema and wide table consumption,
- validate relationship readiness and freshness,
- decide which logic belongs in Databricks SQL vs DAX or Power Query,
- explain Import, DirectQuery and Composite modes,
- simulate incremental refresh with a half-open interval,
- prepare a practical Power BI handoff packet.

## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.

In [0]:
%run ../../setup/00_setup

### Configuration

Confirm the active catalog and schemas before running the Day 2 examples.

In [0]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.

In [0]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")

## 1. Gold Inventory

Definition: a Power BI handoff starts with a concrete object inventory. The BI owner needs object names, row counts and the intended usage of each table.

Expected observation: `fact_sales` and `fact_sales_dashboard` should have the same row count because both are line-grain tables.

In [0]:
%sql
SELECT 'gold.dim_date' AS object_name, 'date dimension' AS purpose, COUNT(*) AS rows FROM gold.dim_date
UNION ALL
SELECT 'gold.dim_customer', 'customer dimension', COUNT(*) FROM gold.dim_customer
UNION ALL
SELECT 'gold.dim_product', 'product dimension', COUNT(*) FROM gold.dim_product
UNION ALL
SELECT 'gold.fact_sales', 'star-schema fact at line grain', COUNT(*) FROM gold.fact_sales
UNION ALL
SELECT 'gold.fact_sales_dashboard', 'wide line-grain dashboard table', COUNT(*) FROM gold.fact_sales_dashboard
ORDER BY object_name

## 2. Semantic Model Definition

A **Power BI semantic model** is the governed layer that contains tables, relationships, measures, formatting and refresh behavior. Reports query the semantic model; the semantic model queries Databricks.

Databricks Gold should provide trusted rows and reusable fields. Power BI should provide report-specific relationships, measures and user interactions.

Expected observation: semantic modeling is a contract decision, not only a UI step in Power BI Desktop.

## 3. Star Schema vs Wide Table Decision

A **star schema** separates facts and dimensions. It is the governed default for reusable semantic models because dimensions can filter many facts consistently.

A **wide table** repeats dimension attributes on each fact row. It is useful for prototypes or narrow dashboard pages, but it can scan more data in DirectQuery and can duplicate business logic.

Professional rule: use `gold.fact_sales` plus dimensions for governed semantic models; use `gold.fact_sales_dashboard` for quick report prototypes or serving tables.

In [0]:
%sql
WITH star AS (
  SELECT ROUND(SUM(CASE WHEN f.is_completed THEN f.line_revenue ELSE 0 END), 2) AS revenue
  FROM gold.fact_sales f
  JOIN gold.dim_customer c ON f.customer_id = c.customer_id
  JOIN gold.dim_product p ON f.product_id = p.product_id
),
wide AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM gold.fact_sales_dashboard
)
SELECT
  star.revenue AS star_schema_revenue,
  wide.revenue AS wide_table_revenue,
  ROUND(star.revenue - wide.revenue, 2) AS revenue_delta
FROM star CROSS JOIN wide

Expected observation: the revenue delta should be zero. The star schema and wide table are different consumption shapes, not different business definitions.

## 4. Relationship Readiness

**Relationship readiness** means dimension keys are unique and fact rows can match those keys. Power BI many-to-one relationships depend on this condition.

An **orphan check** finds fact rows without matching dimensions. Orphans usually become blanks in slicers and can confuse users.

In [0]:
%sql
SELECT 'dim_date.date_key uniqueness' AS check_name, COUNT(*) - COUNT(DISTINCT date_key) AS issue_rows FROM gold.dim_date
UNION ALL
SELECT 'dim_customer.customer_id uniqueness', COUNT(*) - COUNT(DISTINCT customer_id) FROM gold.dim_customer
UNION ALL
SELECT 'dim_product.product_id uniqueness', COUNT(*) - COUNT(DISTINCT product_id) FROM gold.dim_product
UNION ALL
SELECT 'fact_sales -> dim_date orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_date d ON f.order_date = d.date_key
UNION ALL
SELECT 'fact_sales -> dim_customer orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_customer c ON f.customer_id = c.customer_id
UNION ALL
SELECT 'fact_sales -> dim_product orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_product p ON f.product_id = p.product_id

Expected observation: every issue count should be zero before building Power BI relationships.

## 5. Databricks SQL vs DAX vs Power Query

| Logic | Best home | Reason |
|---|---|---|
| joins to conformed dimensions | Databricks Gold | shared, expensive and reusable |
| unknown labels and source quality flags | Databricks Gold | consistent across BI tools |
| additive base fields | Databricks Gold fields | easy to validate and reuse |
| report-specific measures | Power BI DAX | interactive and presentation-aware |
| time intelligence formatting | Power BI DAX | depends on report calendar behavior |
| large row-by-row transformations | Databricks SQL | avoid Power Query work over DirectQuery |
| cosmetic report-only renames | Power BI | does not change shared data contract |

Expected observation: the decision is about ownership. Shared logic belongs closer to Gold; visual logic belongs in the semantic model.

## 6. KPI Dictionary and Starter Measures

A **KPI dictionary** defines each measure in business language before report authors build visuals. Ambiguous filters are the fastest way to create inconsistent dashboards.

A **DAX measure** is a reusable calculation evaluated at query time in the current filter context. Measures should express business logic such as revenue, margin rate or return rate.

In [0]:
%sql
SELECT 'Revenue' AS kpi_name, 'SUM line_revenue where is_completed = true' AS definition, 'Databricks field + DAX measure' AS owner
UNION ALL
SELECT 'Gross Margin', 'SUM line_margin where is_completed = true', 'Databricks field + DAX measure'
UNION ALL
SELECT 'Completed Orders', 'DISTINCTCOUNT order_id where is_completed = true', 'DAX measure'
UNION ALL
SELECT 'Margin Rate %', 'Gross Margin divided by Revenue with divide-by-zero protection', 'DAX measure'
UNION ALL
SELECT 'Return Rate %', 'Returned orders divided by completed plus returned orders', 'DAX measure or certified KPI table' 

Starter DAX measures:

```DAX
Revenue =
CALCULATE (
    SUM ( fact_sales[line_revenue] ),
    fact_sales[is_completed] = TRUE ()
)

Gross Margin =
CALCULATE (
    SUM ( fact_sales[line_margin] ),
    fact_sales[is_completed] = TRUE ()
)

Completed Orders =
CALCULATE (
    DISTINCTCOUNT ( fact_sales[order_id] ),
    fact_sales[is_completed] = TRUE ()
)

Margin Rate % =
DIVIDE ( [Gross Margin], [Revenue] )
```

Expected observation: Databricks supplies governed fields and filters; DAX defines reusable report measures.

## 7. Import, DirectQuery and Composite

![Import vs DirectQuery](../../assets/images/import_vs_directquery.png)

`Import` copies data into Power BI memory during refresh. It is usually fastest for dashboards and safest for cost.

`DirectQuery` sends each interaction back to the SQL Warehouse. It is useful for live requirements, but it can multiply queries through visuals and slicers.

`Composite` mixes Import and DirectQuery tables. It solves mixed freshness requirements, but it increases model complexity.

In [0]:
%sql
SELECT 'Executive dashboard' AS report_area, 'Import' AS recommended_mode, 'fast interactions and scheduled refresh' AS reason
UNION ALL
SELECT 'Operational live monitor', 'DirectQuery', 'freshness requirement is stronger than latency risk'
UNION ALL
SELECT 'Hybrid management report', 'Composite', 'import dimensions and aggregates, query live detail only when needed'
UNION ALL
SELECT 'Ad-hoc analyst exploration', 'Import or Databricks SQL', 'avoid uncontrolled DirectQuery fan-out from many visuals' 

## 8. Incremental Refresh Definition

Incremental refresh partitions imported data by a date or timestamp range. Power BI uses `RangeStart` and `RangeEnd` parameters. The safe pattern is a half-open interval: `>= RangeStart` and `< RangeEnd`.

Expected observation: half-open intervals avoid overlapping rows between adjacent refresh windows.

In [0]:
%sql
WITH refresh_window AS (
  SELECT DATE '2024-01-01' AS range_start, DATE '2024-04-01' AS range_end
)
SELECT
  range_start,
  range_end,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard CROSS JOIN refresh_window
WHERE order_date >= range_start
  AND order_date < range_end
GROUP BY range_start, range_end

## 9. Query Folding Definition

**Query folding** means Power Query can translate transformation steps back to the source query. In Databricks DirectQuery, folding is essential because non-folding transformations can create inefficient or unsupported query plans.

Professional rule: do heavy shaping in Databricks Gold and keep Power Query transformations simple, especially for DirectQuery.

In [0]:
%sql
EXPLAIN FORMATTED
SELECT
  category,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard
WHERE order_date >= DATE '2024-01-01'
  AND order_date < DATE '2025-01-01'
GROUP BY category
ORDER BY revenue DESC

Expected observation: filters and aggregation should appear in the Databricks SQL plan. If a Power Query step cannot fold, redesign it in Gold or in a serving view.

## 10. Freshness and Row Count Checks

Freshness tells report owners how current the data is. Row-count trend checks catch unexpected drops before a scheduled refresh publishes an empty or partial dataset.

In [0]:
%sql
SELECT
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS orders
FROM gold.fact_sales_dashboard

## 11. Connector Walkthrough

![Power BI connection walkthrough](../../assets/images/powerbi_connection_walkthrough.png)

Power BI needs the Databricks SQL Warehouse hostname and HTTP path. Authentication depends on workspace policy: PAT, OAuth or Microsoft Entra ID.

Trainer note: open SQL Warehouses -> Connection details during delivery and show where those two values come from.

## 12. Power BI on Databricks Anti-patterns and Approval Checklists

These are the problems that most often create slow, expensive or inconsistent Power BI reports on Databricks.

| Anti-pattern | Why it hurts | Better pattern |
|---|---|---|
| DirectQuery on a wide line-grain table with many slicers | every interaction fans out into expensive SQL | Import aggregates; reserve DirectQuery for justified live detail |
| importing Silver instead of Gold | report authors recreate business rules independently | use certified Gold or serving objects |
| no date table | time intelligence and incremental refresh become fragile | use `gold.dim_date` and a stable date/timestamp column |
| different Revenue definitions across reports | executives see conflicting KPI totals | publish KPI dictionary and certified measures |
| heavy Power Query transformations over DirectQuery | folding may fail and latency increases | push shaping into Databricks Gold or serving views |
| counting orders as rows | line-grain facts overcount orders | use `COUNT(DISTINCT order_id)` with explicit filters |
| aggregate not reconciled to detail | summary and drill-through disagree | reconcile aggregate totals to the certified detail source |

Dataset readiness checklist:

- grain is documented,
- owner is known,
- row counts and freshness are validated,
- relationships are many-to-one ready,
- KPI rules are approved,
- PII and restricted fields are handled,
- refresh expectation is documented.

DirectQuery approval checklist:

- live freshness is a real business requirement,
- source object is narrow or aggregated enough,
- expected visual count and slicer behavior are controlled,
- SQL Warehouse cost guardrails are agreed,
- Query History will be reviewed after publishing,
- Import or Composite was considered first.

Expected observation: DirectQuery is a design exception, not the default answer to freshness concerns.

## 13. Questions to Ask the Databricks/Data Team

Before building or certifying a Power BI semantic model, ask these questions and capture the answers in the handoff packet.

| Question | Why Power BI cares |
|---|---|
| What is the grain of each source object? | prevents duplicated facts and wrong relationships |
| Who owns the Gold or serving object? | identifies who approves schema and KPI changes |
| What is the freshness SLA? | drives Import refresh schedule and DirectQuery justification |
| Is DirectQuery allowed for this object? | controls SQL Warehouse cost and user experience |
| Which KPI definitions are certified? | prevents conflicting DAX measures across reports |
| Does the data contain PII or restricted fields? | affects row/column security and sharing |
| What is the refresh failure escalation path? | prevents silent stale reports |

Expected observation: this is the practical bridge between a BI developer and the Databricks platform team.

## 14. Power BI Handoff Packet

A professional handoff is a packet of decisions, not just a connection string.

| Handoff item | Required decision |
|---|---|
| Connection details | SQL Warehouse hostname and HTTP path from Databricks UI |
| Authentication | PAT, OAuth or Entra ID pattern chosen by workspace policy |
| Source option | star schema for governed model, dashboard table for quick prototype |
| Relationship map | fact date/customer/product keys to unique dimensions |
| Model mode | Import by default, DirectQuery only with explicit freshness need |
| Refresh expectation | schedule, owner and freshness SLA |
| Cost guardrails | warehouse size, auto-stop, visual count and DirectQuery limits |
| Measure ownership | which measures are DAX-owned vs Gold-owned |

Expected observation: this table is the conversation checklist between Databricks and Power BI owners.

## 15. Performance Analyzer Awareness

Power BI Performance Analyzer shows which visuals trigger slow DAX queries or source queries. For Databricks DirectQuery, pair it with SQL Warehouse Query History.

Expected observation: a slow report page is investigated from both sides: Power BI visual fan-out and Databricks SQL execution profile.

## Summary and Workshop Handoff

You now have the semantic model vocabulary and the contract for Workshop 2: build smaller, refresh-ready, performance-aware objects for Power BI without changing the Day 1 Gold model.

---
## 16. Live Demo: Power BI Desktop Connection (15 min)

**Cel:** Pokaz na zywo jak polaczyc Power BI Desktop z SQL Warehouse i zbudowac pierwszy raport z Gold layer.

**Trainer note:** Ponizsze kroki wykonujesz live na projektorze. Uczestnicy obserwuja i robia notatki/screenshoty.

### Krok 1: Pobranie Connection Details

**Sciezka w UI:** SQL Warehouses > [nazwa warehouse] > Connection details

Potrzebne wartosci:
* **Server hostname:** np. `adb-1234567890.1.azuredatabricks.net`
* **HTTP path:** np. `/sql/1.0/warehouses/abc123def456`

Trainer note: pokaz to na zywo w workspace. Uczestnicy robia screenshoty.

### Krok 2: Power BI Desktop - Get Data

**Sciezka w PBI:** Get Data > Azure > Azure Databricks

| Pole | Wartosc | Uwagi |
|------|---------|-------|
| Server Hostname | skopiowany z Connection Details | BEZ `https://` |
| HTTP Path | skopiowany z Connection Details | cala sciezka |
| Data Connectivity mode | **Import** (domyslnie) | Pokaz roznice ponizej |

### Import vs DirectQuery - Decision Table

| Kryterium | Import | DirectQuery |
|-----------|--------|-------------|
| Odswiezanie danych | Scheduled refresh (np. 4x/dzien) | Na zywo przy kazdej interakcji |
| Performance raportu | Szybki (dane w pamieci PBI) | Wolniejszy (query do warehouse) |
| Koszt SQL Warehouse | Tylko podczas refresh | Przy kazdym uzyciu raportu |
| Swiezosc danych | Opozniona (do nastepnego refresh) | Real-time |
| Max rozmiar danych | 1 GB (Pro) / 100 GB (Premium) | Bez limitu |
| Typowe uzycie | Executive dashboards, raporty dzienne | Live monitoring, duze datasety |

**Rekomendacja dla wiekszosci przypadkow:** Import z scheduled refresh co 4-8h.

**DirectQuery tylko gdy:** dane MUSZA byc real-time LUB dataset przekracza limit Import.

### Krok 3: Wybor tabel i budowa modelu

**Demo flow:**
1. Navigator > wybierz catalog `retailhub_trainer` > schema `gold`
2. Zaznacz: `dim_customer`, `dim_product`, `dim_date`, `fact_sales`
3. Load (Import)

### Krok 4: Relationships

PBI powinien automatycznie wykryc relacje (bo klucze sa unikalne w dims):
* `fact_sales.customer_id` -> `dim_customer.customer_id` (Many:1)
* `fact_sales.product_id` -> `dim_product.product_id` (Many:1)
* `fact_sales.order_date` -> `dim_date.date_key` (Many:1)

**UWAGA:** Jesli PBI NIE wykryje relacji automatycznie = problem z grain lub orphan keys. Pokaz ze Unknown member (z Workshop 1) rozwiazuje ten problem.

### Krok 5: Pierwsza wizualizacja (2 min)

1. Stacked bar: Revenue by Category
2. Slicer: Region
3. Card: Total Revenue

**Punkt do podkreslenia:** Wszystko dziala bo Gold layer ma poprawny grain i zero orphanow.

### Krok 6: Incremental Refresh (opcjonalny pokaz)

W Power BI Service mozna skonfigurowac Incremental Refresh:
1. Dodaj parametry `RangeStart` i `RangeEnd` (DateTime) w Power Query Editor
2. Filtruj fact po `order_datetime >= RangeStart AND order_datetime < RangeEnd`
3. PPM na tabele > Incremental refresh > Archive 2 lata, Incremental 30 dni
4. Publish do PBI Service

**Dlaczego `v_fact_sales_incremental` z Workshop 2 jest wazny:**
* View z half-open interval umozliwia query folding
* PBI Service skanuje TYLKO nowe dane
* Bez tego: full refresh = wiecej kosztow SQL Warehouse

---
## 17. Live Demo: Delta Sharing (15 min)

### Czym jest Delta Sharing?

Otwarty protokol do bezpiecznego udostepniania danych POZA organizacje bez kopiowania.

| Cecha | Delta Sharing | Tradycyjne podejscie |
|-------|--------------|---------------------|
| Kopiowanie danych | NIE - odbiorca czyta z Twojego storage | TAK - eksport CSV/Parquet |
| Governance | Pelna kontrola: co, komu, kiedy | Brak po wyslaniu pliku |
| Aktualnosc | Zawsze najnowsza wersja | Stale od momentu eksportu |
| Format | Otwarty (Parquet over REST) | Rozne |
| Odbiorcy | Power BI, Pandas, Spark, Tableau, dowolny klient | Tylko co obsluguje format |

### Scenariusze uzycia

* Udostepnienie Gold KPI partnerowi biznesowemu
* Centrala udostepnia dane oddzialom (rozne UC catalogi)
* Zespol data science konsumuje z Power BI workspace
* Compliance: audytor dostaje read-only widok bez kopiowania

In [0]:
%sql
-- ============================================================
-- DELTA SHARING: Live Demo (trainer only)
-- ============================================================
-- UWAGA: Wymaga uprawnien CREATE SHARE na metastore level
-- Uczestnicy obserwuja - nie wykonuja samodzielnie

-- Krok 1: Utworz Share
CREATE SHARE IF NOT EXISTS retailhub_partner_share
COMMENT 'Gold KPI tables shared with external Power BI consumers';

-- Krok 2: Dodaj tabele do Share
ALTER SHARE retailhub_partner_share ADD TABLE gold.fact_sales_dashboard_monthly;
ALTER SHARE retailhub_partner_share ADD TABLE gold.dim_customer;
ALTER SHARE retailhub_partner_share ADD TABLE gold.dim_product;

-- Krok 3: Sprawdz zawartosc Share
SHOW ALL IN SHARE retailhub_partner_share;

In [0]:
%sql
-- Krok 4: Utworz Recipient (zewnetrzny odbiorca)
CREATE RECIPIENT IF NOT EXISTS partner_analytics_team
COMMENT 'External analytics team consuming Gold KPIs via Power BI';

-- Krok 5: Nadaj dostep
GRANT SELECT ON SHARE retailhub_partner_share
TO RECIPIENT partner_analytics_team;

-- Krok 6: Pobierz activation link
-- W UI: Sharing > Recipients > partner_analytics_team > Activation link
-- Odbiorca uzywa tego linku do polaczenia z Power BI lub innym narzedziem

SHOW RECIPIENTS;

### Jak odbiorca konsumuje Delta Share w Power BI?

| Opcja | Kiedy uzywac | Wymagania |
|-------|-------------|----------|
| **A - Databricks connector** | Odbiorca ma dostep do workspace | Konto Databricks, Get Data > Azure Databricks |
| **B - Delta Sharing connector** | Odbiorca ZEWNETRZNY bez Databricks | PBI Desktop + .share credentials file |
| **C - Fabric Dataflow Gen2** | Organizacja uzywa Fabric | Fabric capacity, Endpoint URL + Bearer token |

**Opcja B szczegolowo (najpopularniejsza dla partnerow):**
1. Odbiorca pobiera activation link z Databricks UI
2. Zapisuje credentials file (.share)
3. W PBI: Get Data > Delta Sharing
4. Wskazuje na .share file
5. Wybiera tabele z listy shared objects

### Key Takeaways

* Delta Sharing = udostepnianie BEZ kopiowania (governance zachowana)
* Odbiorca nie potrzebuje Databricks (otwarty protokol)
* Idealne dla: partnerow, audytorow, zespolow w innym workspace
* Laczy sie naturalnie z Power BI przez dedykowany connector